# Apache Flink

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/flink/flink.ipynb)

Apache Flink is a distributed, stateful stream processor with true event-time semantics, watermarks, keyed state, checkpointing, and exactly-once connectors. It runs the same operators over unbounded streams and bounded (batch) inputs.

Flink itself needs a JVM cluster, so the runnable cells below simulate the core mechanics - windows, watermarks, late data, keyed state, skew, and checkpoint recovery - in plain Python, and PyFlink/Java/SQL snippets are shown as code blocks.

## When would you choose Flink over Spark Structured Streaming?

**The real decision is the execution model.** Flink is a pipelined, record-at-a-time streaming runtime. Each operator runs continuously, and state lives in the operator, protected by periodic checkpoints. Spark Structured Streaming runs micro-batches: every trigger it plans a batch, reads new offsets, computes, and commits. Latency therefore has a floor of one trigger interval, and state is carried between batches in a state store.

Pick Flink when per-event latency matters, when event-time correctness with late data matters, or when state is large and long lived. Pick Spark when you already run Spark, when you want one API for batch and stream, or when a one-minute micro-batch already meets the SLA.

Worked example: an ad-billing pipeline needs a p99 latency under 500 ms and must attribute clicks that arrive up to two minutes late. Flink with event time and 2-minute watermarks fits. An hourly revenue rollup does not need Flink; Spark or dbt is cheaper to operate.

**Common mistake:** assuming Flink is always faster. For batch-like, high-throughput ETL, Spark's batch engine often wins on cost per record because it amortizes scheduling and serialization across a whole batch.

## What are event time, processing time, and watermarks in Flink?

**Three clocks.** Event time is the timestamp inside the record, set by the producer. Ingestion time is when the record entered Flink. Processing time is the wall clock of the operator that handles it.

A watermark is a heuristic promise: no more events with a timestamp older than T should arrive. It is not enforced by the source. In event time, a window fires when the watermark passes the window end. The watermark at an operator is the minimum of its inputs, so one idle partition can hold everything back.

Worked example: records carry `event_ts`. Assign `WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(5))`. A window `[10:00, 10:01)` fires when the watermark reaches 10:01, roughly five seconds after the newest event seen.

**Common mistake:** using processing time for anything you must replay or audit. Processing-time results depend on when the job ran. Event time is deterministic, but you pay for timestamp assignment and watermark tracking, and you must decide what happens to records that arrive after the window fired.

## What is Flink and how is a job structured?

Flink is a distributed, stateful stream-processing framework that treats batch as a bounded stream. A job is a dataflow of `source -> transformation -> sink`, and the same runtime handles unbounded (streaming) and bounded (batch) input.

The main APIs are the DataStream API (Java, Scala, Python), the Table API and SQL, plus domain libraries such as CEP and FlinkML. Flink provides event-time processing, watermarks, keyed state, checkpointing, and exactly-once semantics.

```python
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
env.from_collection(["hello", "flink"]).print()
env.execute("flink-intro")
```

**Mental model of a job.** A Flink program is a dataflow graph: one or more sources, a chain of transformations, and one or more sinks. Operators can be parallel, and the ones after a `key_by` or `rebalance` are partitioned across the network. Nothing runs until `env.execute()` is called.

A saved program is compiled twice. The client builds a JobGraph, which is submitted to the JobManager. The JobManager builds the ExecutionGraph, which expands each operator into parallel subtasks and assigns them to slots. The physical execution is a set of tasks, each of which may contain several chained operators.

**Common mistake:** expecting side effects to happen while building the graph. Transformations are lazy; a `print` in a `map` runs only after `execute()`.

**How to read a job:** source -> map/filter (likely chained into the source task) -> key_by/window (new task, network shuffle) -> sink. Chained operators share a thread and a slot; a key_by breaks the chain.

## What are the differences between Flink and Spark Streaming?

| Aspect | Spark Structured Streaming | Flink |
|---|---|---|
| Model | micro-batch | true record-at-a-time streaming |
| Latency | sub-second to seconds | milliseconds |
| Time semantics | event time with watermarks | event time, watermarks, three time notions |
| State | limited, epoch-based | large keyed state, RocksDB |
| Exactly-once | checkpoint + idempotent sink | checkpoint + two-phase-commit sinks |
| Batch | Spark SQL engine | bounded streams on the same runtime |

## What are the commonly used operators in Flink?

```python
stream = env.from_source(source, watermark_strategy, "orders")

mapped = stream.map(lambda o: enrich(o))
flat = stream.flat_map(lambda o: o["items"])
filtered = stream.filter(lambda o: o["amount"] > 0)
keyed = filtered.key_by(lambda o: o["customer_id"])
counts = keyed.map(CountByKey())
joined = keyed.connect(other).key_by(lambda a: a["k"], lambda b: b["k"]).process(JoinFn())
```

**Operator map.** Sources (`from_source`, Kafka, files), stateless transforms (`map`, `flat_map`, `filter`), partitioning (`key_by`, `rebalance`, `broadcast`, `forward`), stateful transforms (`reduce`, `aggregate`, `process`, `window`), and sinks (`sink_to`, Kafka, JDBC, files). `key_by` repartitions and always breaks operator chaining.

Chaining is why a simple pipeline is cheap. A source, a `map`, and a `filter` with the same parallelism and only forward edges are fused into one task, one thread, and one slot, so records never cross the network.

Worked example: `source -> map -> filter -> key_by -> window -> reduce -> sink` compiles into roughly two tasks: the fused source/map/filter, and the keyed window/reduce/sink chain.

**Common mistake:** putting a `rebalance` or `key_by` in front of every cheap operator and turning a single-threaded pipeline into a network-bound one. Only repartition when you need key locality or even distribution.

## How does parallelism work in Flink?

```python
env.set_parallelism(8)                                                 # job-wide default
stream.key_by(lambda o: o["cust"]).map(Enrich()).set_parallelism(16)   # operator override
```

```bash
# CLI override at submit time
flink run -p 16 -c com.acme.OrdersJob orders.jar
```

**Parallelism is per operator.** Each operator has a parallelism, which is the number of parallel subtasks. The effective value follows precedence: an operator-level setting wins, then the environment setting, then the client or CLI `-p`, then `parallelism.default`. The maximum parallelism ever used must be reflected in the slots and in `state.backend` configuration.

Sources are bounded by the input. A Kafka source with 12 partitions can use at most 12 useful source subtasks; any extra subtasks sit idle. Downstream keyed operators can use a higher parallelism because they repartition.

```yaml
parallelism.default: 16
taskmanager.numberOfTaskSlots: 4
# 4 TaskManagers provide 16 slots
```

**Common mistake:** setting `key_by` parallelism to 1, which funnels every key through one subtask and creates a permanent bottleneck. Also common: raising parallelism without raising Kafka partitions, so the extra subtasks do nothing.

## What happens when a Flink job fails and restarts?

```python
from pyflink.common import Configuration
from pyflink.datastream import StreamExecutionEnvironment

config = Configuration()
config.set_string("restart-strategy.type", "fixed-delay")
config.set_string("restart-strategy.fixed-delay.attempts", "4")
config.set_string("restart-strategy.fixed-delay.delay", "10 s")
env = StreamExecutionEnvironment.get_execution_environment(config)
# other types: "failure-rate", "exponential-delay" (the default when checkpointing is on), "none"
```

Set the restart strategy through configuration. The older `RestartStrategies` API was deprecated in
Flink 1.19 and removed in Flink 2.0.

**Restart is a state recovery, not a cold start.** On failure, the affected tasks are cancelled and the JobManager restarts them from the latest completed checkpoint. Sources rewind to the offsets stored in that checkpoint, operator state is restored from the snapshot, and processing resumes. The rest of the pipeline may keep running if only a region failed.

The restart strategy decides how often and how fast restarts happen. `fixed-delay` waits a fixed time between attempts. `failure-rate` restarts only if failures stay under a threshold. Exponential delay backs off under repeated failures.

```yaml
restart-strategy: fixed-delay
restart-strategy.fixed-delay.attempts: 3
restart-strategy.fixed-delay.delay: 10s
execution.checkpointing.tolerable-failed-checkpoints: 3
```

**Common mistake:** a permanent failure plus aggressive restarts creates a crash loop that floods the logs and never recovers. Use a bounded restart strategy and alert on repeated failures.

## What is Flink's distributed cache?

**One copy per TaskManager.** The distributed cache registers a file with the job. When a task starts, the TaskManager downloads the file once into its working directory and every subtask on that worker reads the local copy. This is how small static lookup files are shipped with a job.

In PyFlink you register the file with `env.register_cached_file(path, name)` and read it from the task's working directory. In Java you call `getRuntimeContext().getDistributedCache().getFile(name)`.

**Common mistake:** expecting the cache to refresh while the job runs. Files are fetched at task startup, so changing the file has no effect until the job restarts. For values that must change at runtime, use broadcast state or a lookup table, not the distributed cache.

## What is broadcast state?

**Broadcast is a stream, not a variable.** You connect a small stream of rules or metadata to a main stream using `broadcast()` and `connect()`. A `BroadcastProcessFunction` reads the broadcast side and can emit enriched records from the main side. Each subtask stores its own copy of the broadcast state.

Broadcast state is operator state, not keyed state. It is held on heap, so it is meant for small, slowly changing data such as currency rates or feature flags. Broadcast state descriptors must match by name and type between the broadcast and the process function.

```python
# sketch: rules broadcast to every parallel subtask
# rules = env.from_source(...)
# main.connect(rules.broadcast(descriptor)).process(BroadcastEnrich())
```

**Common mistake:** broadcasting a large table. Every subtask holds a full copy, so memory grows with parallelism. For large dimension tables use a lookup join with a cached external table instead.

## What windows does Flink support?

```python
from pyflink.common import Time
from pyflink.datastream import window

keyed = stream.key_by(lambda e: e["user_id"])
keyed.window(window.TumblingEventTimeWindows.of(Time.minutes(5))).aggregate(SumAmount())
keyed.window(window.SlidingEventTimeWindows.of(Time.minutes(15), Time.minutes(5))).process(TopN())
keyed.window(window.EventTimeSessionWindows.with_gap(Time.minutes(10))).process(Sessionize())
keyed.count_window(100).aggregate(SumAmount())   # count window
```

The assignment rules behind those window types, simulated:

In [ ]:
# Window assignment in plain Python: which windows does each event belong to?
from collections import defaultdict

MIN = 60  # seconds


def tumbling(ts: int, size: int) -> list[tuple[int, int]]:
    start = ts - ts % size
    return [(start, start + size)]


def sliding(ts: int, size: int, slide: int) -> list[tuple[int, int]]:
    last_start = ts - ts % slide
    return [(s, s + size) for s in range(last_start - size + slide, last_start + 1, slide) if s <= ts < s + size]


def sessions(timestamps: list[int], gap: int) -> list[tuple[int, int]]:
    out: list[list[int]] = []
    for ts in sorted(timestamps):
        if out and ts - out[-1][1] < gap:
            out[-1][1] = ts                      # extend the open session
        else:
            out.append([ts, ts])
    return [(start, end + gap) for start, end in out]   # a session closes `gap` after its last event


clicks = [(0, 10), (2 * MIN, 5), (4 * MIN, 7), (6 * MIN, 1), (30 * MIN, 4)]   # (event_ts, amount)

tumbling_sums = defaultdict(int)
for ts, amount in clicks:
    for window in tumbling(ts, 5 * MIN):
        tumbling_sums[window] += amount

print("tumbling 5m:", dict(tumbling_sums))
print("event at 16m is in sliding 15m/5m windows:", sliding(16 * MIN, 15 * MIN, 5 * MIN))
print("sessions with a 10m gap:", sessions([ts for ts, _ in clicks], 10 * MIN))
assert dict(tumbling_sums) == {(0, 300): 22, (300, 600): 1, (1800, 2100): 4}
assert sliding(16 * MIN, 15 * MIN, 5 * MIN) == [(300, 1200), (600, 1500), (900, 1800)]  # size/slide copies          # size / slide copies per event
assert sessions([ts for ts, _ in clicks], 10 * MIN) == [(0, 960), (1800, 2400)]

**A window is a way to group infinite streams by time or count.** Time windows come in tumbling (fixed, non-overlapping), sliding (fixed size, fixed slide), session (gaps of inactivity), and global (one window per key, closed by a trigger). Windows can be keyed or non-keyed; non-keyed windows use `window_all` and run at parallelism 1.

Each window creates state per key. That state is what makes windows expensive: many keys times many open windows. Allowed lateness keeps the window state alive after the window fires so late records can update results.

Worked example: `stream.key_by(lambda e: e["region"]).window(TumblingEventTimeWindows.of(Time.minutes(5)))` keeps one pane per region per five-minute interval, and emits when the watermark passes the end.

**Common mistake:** keying by a high-cardinality field such as a user id and using long sliding windows. The state can explode far beyond the input rate.

## Where does Flink keep state?

State backends decide where keyed state lives and how checkpoints are written. `HashMapStateBackend` keeps state on the JVM heap (fast, memory-bound), while `EmbeddedRocksDBStateBackend` spills to local disk and supports incremental checkpoints (large state). See the dedicated state backend question for configuration.

**Keyed state is scoped by key; operator state is scoped by subtask.** Keyed state descriptors (`ValueState`, `ListState`, `MapState`, `ReducingState`, `AggregatingState`) live in the backend and are partitioned by key. Operator state, including `ListState` used as union state and `BroadcastState`, is bound to the subtask and redistributed on rescale.

Which backend you choose changes the cost model. The heap backend keeps state in JVM objects and writes a full snapshot each checkpoint. RocksDB keeps state in a local embedded database and can write only changed files when incremental checkpoints are on.

**Common mistake:** treating state as free. A `MapState` with millions of keys on the heap causes long GC pauses and slow checkpoints. Move it to RocksDB, add state TTL, or redesign the key.

## What is a watermark?

```python
from pyflink.common import Duration, WatermarkStrategy
from pyflink.common.watermark_strategy import TimestampAssigner

class EventTimestamps(TimestampAssigner):
    def extract_timestamp(self, event, record_timestamp):
        return event["event_ts"]

wm = (
    WatermarkStrategy
    .for_bounded_out_of_orderness(Duration.of_seconds(5))
    .with_timestamp_assigner(EventTimestamps())
)
stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "kafka-orders")
stream = stream.assign_timestamps_and_watermarks(wm)
```

How the watermark advances across partitions and fires windows:

In [ ]:
# Bounded out-of-orderness watermarks across two partitions, and why an idle partition stalls them.
OUT_OF_ORDERNESS = 5     # seconds
WINDOW = 60


def run(events: list[tuple[int, int]], idle_timeout: int | None = None) -> list[int]:
    """events: (partition, event_ts) in arrival order; returns the window ends that fired."""
    max_ts = {0: None, 1: None}
    last_seen = {0: 0, 1: 0}
    fired, next_end = [], WINDOW
    for step, (partition, ts) in enumerate(events):
        max_ts[partition] = ts if max_ts[partition] is None else max(max_ts[partition], ts)
        last_seen[partition] = step
        active = [p for p in max_ts if max_ts[p] is not None and
                  (idle_timeout is None or step - last_seen[p] <= idle_timeout)]
        if len(active) < len(max_ts) and idle_timeout is None:
            continue                                      # a partition with no data holds everything
        watermark = min(max_ts[p] for p in active) - OUT_OF_ORDERNESS
        while watermark >= next_end:                      # window [end-60, end) is complete
            fired.append(next_end)
            next_end += WINDOW
    return fired


only_partition_0 = [(0, t) for t in (10, 50, 70, 130, 190)]
print("partition 1 silent, no idleness:", run(only_partition_0))
print("partition 1 silent, with idleness:", run(only_partition_0, idle_timeout=1))
both = [(0, 10), (1, 12), (0, 70), (1, 66), (0, 130), (1, 128)]
print("both partitions active:", run(both))
assert run(only_partition_0) == []                        # the stall `with_idleness` fixes
assert run(only_partition_0, idle_timeout=1) == [60, 120, 180]
assert run(both) == [60, 120]

**Watermarks flow like records with a timestamp.** A source's watermark strategy emits periodic watermarks. An operator with several inputs takes the minimum, because it can only advance as far as its slowest input. Watermarks never go backwards.

`for_bounded_out_of_orderness` trades latency for completeness: a larger delay means more late records are still accepted but windows fire later. `for_monotonous_timestamps` is the cheapest and assumes sorted input. `with_idleness` lets a partition that stops producing data be excluded so it cannot hold the watermark back.

```sql
CREATE TABLE orders (
  order_id BIGINT,
  region STRING,
  amount DECIMAL(10,2),
  order_ts TIMESTAMP(3),
  WATERMARK FOR order_ts AS order_ts - INTERVAL '5' SECOND
) WITH ('connector' = 'kafka', 'topic' = 'orders');
```

**Common mistake:** setting the watermark delay too small on noisy data. Records arriving after the window fires are dropped unless you configure allowed lateness or a side output.

## What is DataStream API in Apache Flink?

```python
from pyflink.common import Types, WatermarkStrategy
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "orders")

result = (
    stream.map(lambda o: (o["region"], o["amount"]), output_type=Types.TUPLE([Types.STRING(), Types.DOUBLE()]))
    .key_by(lambda t: t[0])
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
)
result.print()
env.execute("regional-revenue")
```

**The DataStream API is the typed, stateful core.** A `DataStream` is an immutable handle; each call returns a new stream and records an operator in the graph. `KeyedStream` comes from `key_by` and is the only place keyed state exists. `ConnectedStreams` and `BroadcastStream` are the other special streams.

The API exposes process functions for full control: `KeyedProcessFunction` gives you timers, keyed state, and side outputs. Simpler operators such as `map` and `filter` are shortcuts over the same runtime.

Worked example: a fraud rule needs to see the previous transaction per account. `key_by(account).process(MarkSuspicious())` keeps a `ValueState` per account, which a stateless `map` could never do because it does not know the key.

**Common mistake:** using `process` where a built-in `reduce` or `aggregate` would do. Aggregate functions keep bounded state; a `process` function that buffers every record grows without limit unless you add TTL or clear state.

## What is Apache Flink Table API?

**One declarative query, two runtime modes.** The Table API and SQL describe a query over tables. In streaming mode the query is continuous and produces a changelog; in batch mode it runs once. The planner converts the query into the same kind of dataflow the DataStream API builds, with optimizations such as filter pushdown and projection pruning.

Streaming tables are dynamic. A query can produce insert-only rows or a changelog with updates and deletes. Sinks that can handle updates need a primary key; insert-only sinks such as files or Kafka append topics cannot consume retractions.

```sql
CREATE TABLE orders (
  order_id BIGINT,
  region STRING,
  amount DECIMAL(10,2),
  order_ts TIMESTAMP(3),
  WATERMARK FOR order_ts AS order_ts - INTERVAL '5' SECOND
) WITH ('connector' = 'kafka', 'topic' = 'orders',
        'properties.bootstrap.servers' = 'broker:9092',
        'format' = 'json');

SELECT region, SUM(amount) FROM orders GROUP BY region;
```

**Common mistake:** sending an updating aggregate to an append-only sink. The job fails or silently emits only some changes. Use an upsert sink or a windowed query that is insert-only.

## What are the use cases of Flink?

**The pattern behind the use cases.** Flink pays off when the pipeline is stateful, event-time sensitive, and expected to run continuously: real-time analytics, continuous ETL and CDC, event-driven applications, anomaly and fraud detection, and feature computation for online models.

A useful test: if the answer changes depending on when late data arrives, or if you need per-key memory across events, Flink's model fits. If the answer is a periodic batch report, it does not.

**Common mistake:** using Flink as a message queue. Operators can buffer briefly, but Flink is not durable storage; if the job is down for a long time, the source must still hold the data to replay.

## What are bounded and unbounded streams?

**Bounded data has an end; unbounded data does not.** The distinction affects scheduling and state. A bounded job can finish, release state at stage boundaries, and run in batch mode. An unbounded job runs until stopped and must checkpoint to recover.

The same program can often handle both. The source defines the nature of the data; `execution.runtime-mode` defines how Flink executes it. Batch mode on bounded input enables blocking shuffles and resource reuse.

**Common mistake:** assuming a Kafka source is always unbounded. It can be bounded by `set_bounded(OffsetsInitializer.latest())`, which makes it a finite read, often used for backfills and tests.

## How do you monitor a running Flink job?

```bash
curl -s http://jobmanager:8081/jobs/overview
curl -s http://jobmanager:8081/jobs/<jobId>/checkpoints | head
curl -s http://jobmanager:8081/jobs/<jobId>/metrics?get=numRecordsOut,currentInputWatermark
```

## How does Flink provide fault tolerance?

**Checkpoint and replay.** Sources inject barriers, operators snapshot state when the barrier passes, and the JobManager marks the checkpoint complete once all sinks acknowledge. On failure, tasks restore that state and the sources rewind to the stored offsets. Everything between two checkpoints is replayed.

Exactly-once holds when the source is replayable and the sink either supports two-phase commit or is idempotent. The state backend decides how the snapshot is written: heap state is copied in full; RocksDB can write only changed files.

```java
env.enableCheckpointing(60_000, CheckpointingMode.EXACTLY_ONCE);
```

**Common mistake:** assuming checkpointing alone gives exactly-once end to end. An external side effect such as an email or a non-transactional JDBC write will still duplicate on replay.

## What does the JobManager do?

**Coordination only.** The JobManager accepts the JobGraph, builds and schedules the ExecutionGraph, allocates slots through the ResourceManager, coordinates checkpoints, and drives recovery on failure. It also serves the UI and REST API.

It does not process records. It holds job metadata in memory, and for high availability writes it to a durable store so a standby can recover. Each job has its own JobMaster inside the JobManager process.

**Common mistake:** giving the JobManager a large heap "for state". Keyed state lives in TaskManagers; the JobManager only keeps scheduling and checkpoint metadata. Size it from the number of jobs and operators, not from state size.

## What does a TaskManager do?

**Work and state.** A TaskManager is a JVM that owns a set of slots, provides managed memory, runs tasks, and stores operator state for those tasks. It exchanges data with other TaskManagers over the network and writes checkpoint data to durable storage.

TaskManager memory is split into framework heap, task heap, managed memory (used by RocksDB and batch operators), and network memory. `taskmanager.memory.process.size` sets the total; the fractions decide the split.

```yaml
taskmanager.memory.process.size: 4096m
taskmanager.memory.managed.fraction: 0.4
taskmanager.numberOfTaskSlots: 4
```

**Common mistake:** packing too many slots into one TaskManager. They share the JVM, its GC, its network buffers, and its local disk, so a heavy slot slows all the others.

## What can you do with state?

**Read, update, expire, and snapshot it.** Keyed state is accessed through descriptors inside a `ProcessFunction` or `RichFunction`. You can read and write values, iterate map and list state, clear a key's state, and attach state TTL so old keys disappear.

Beyond the operator, state is visible through checkpoint and savepoint files. The State Processor API can read a savepoint into a dataset, modify it, and write a new savepoint, which is how you bootstrap state for a new job.

**Common mistake:** never clearing state. A per-user `ValueState` that is never removed grows forever, and checkpoints grow with it. Add TTL or clear state when the key is finished.

```python
# StateTtlConfig is attached to the descriptor's config, not the backend
# ttl = StateTtlConfig.new_builder(Time.days(7)).set_update_type(StateTtlConfig.UpdateType.OnCreateAndWrite).build()
```

## How do you join streams in Flink?

```java
// Tumbling window inner join: join on key and time window
left.join(right)
    .where(a -> a.key)
    .equalTo(b -> b.key)
    .window(TumblingEventTimeWindows.of(Time.minutes(5)))
    .apply(new JoinFunction<Event, Event, Joined>() {
        @Override
        public Joined join(Event a, Event b) { return new Joined(a, b); }
    });
```

These joins are part of the Java/Scala DataStream API; PyFlink does not expose `join`, `coGroup`, or `intervalJoin`.

**Three ways to join streams.** Window joins pair records from two streams that fall in the same window and share a key; they are time-bounded but hold all records of the window. Interval joins pair each record with records from the other stream inside a time interval relative to its timestamp, which bounds state by the interval. Regular joins keep state indefinitely unless you add a TTL.

In Flink SQL, window joins are written as two windowed table functions on a common window and joined on `window_start`, `window_end`, and the key. Lookup joins consult an external table per record and are the usual choice for dimension data.

```sql
SELECT o.order_id, c.name
FROM TABLE(TUMBLE(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' MINUTE)) o
JOIN TABLE(TUMBLE(TABLE clicks, DESCRIPTOR(click_ts), INTERVAL '1' MINUTE)) c
  ON o.order_id = c.order_id AND o.window_start = c.window_start;
```

**Common mistake:** a regular streaming join between two busy streams. State for both sides grows without bound; use a window or interval join, or set `table.exec.state.ttl`.

Interval join (Java), pairing records whose timestamps are within five seconds:

```java
KeyedStream<Event, String> keyed1 = ds1.keyBy(o -> o.getString("key"));
KeyedStream<Event, String> keyed2 = ds2.keyBy(o -> o.getString("key"));
// right.ts - 5s <= left.ts <= right.ts + 5s
keyed1.intervalJoin(keyed2)
    .between(Time.seconds(-5), Time.seconds(5))
    .process(new ProcessJoinFunction<Event, Event, Joined>() {
        @Override
        public void processElement(Event a, Event b, Context ctx, Collector<Joined> out) {
            out.collect(new Joined(a, b));
        }
    });
```

## Which window functions does Flink SQL offer?

**Two families.** Grouped window aggregation (`GROUP BY window_start, window_end`) reduces each window to few rows. Window table-valued functions such as `TUMBLE`, `HOP`, and `CUMULATE` produce those window columns. Over aggregation (`ROW_NUMBER() OVER (...)`) computes rankings and running values within a partition.

Streaming over-aggregation needs an ordered, time-bounded input to be useful. `ROW_NUMBER` over a window is how TopN is written; without a window, the rank can change forever and the state never shrinks.

```sql
SELECT * FROM (
  SELECT order_id, region, amount,
         ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) AS rn
  FROM orders
) WHERE rn <= 3;
```

**Common mistake:** running a global `ROW_NUMBER` TopN over an unbounded stream. The job keeps every row to maintain the ranking; add a window or an event-time bound, or the state grows without limit.

## How does a KeyedProcessFunction work?

**One record at a time, per key.** `KeyedProcessFunction` receives each element together with a context that exposes the current key, keyed state, timers, and side outputs. `process_element` returns nothing; output goes through `collect`.

Timers are registered against event or processing time. Each timer is identified by timestamp plus key, so registering the same timestamp for a key twice fires once. `on_timer` runs when the clock passes it, and it is where you flush buffered state or emit session results.

**Common mistake:** registering a new timer on every record without deduplicating. Thousands of duplicate timers slow the job; check state, or let the runtime coalesce timers at the same timestamp.

## How do you enrich a stream with dimension data?

Dimension enrichment from a database or object store is usually done one of four ways, in increasing order of freshness and cost:

+ **Async I/O**: query an external system per record without blocking the operator thread.
+ **Broadcast state**: broadcast a small dimension to every task.
+ **Async I/O plus cache**: add a TTL cache in front of the lookup.
+ **Scheduled refresh**: load the dimension in `open()` and refresh on a timer.

PyFlink does not expose `RichAsyncFunction`/`AsyncDataStream`; use the Java DataStream API above or a Table API lookup join.

**Match the join to the data size and freshness.** Small, slow-changing dimensions go into broadcast state or a lookup join with a cache. Large or versioned dimensions use a temporal join against a versioned table, where each fact is joined to the dimension row that was current at the fact's event time.

For high lookup volume, async I/O is the key optimization. A synchronous lookup blocks the task on every record; `AsyncDataStream` lets many lookups be in flight while other records keep processing.

**Common mistake:** a synchronous lookup join against a remote database under load. Latency per record times records per second quickly exceeds the task capacity, and backpressure spreads upstream. Use a cache or async I/O.

```sql
SELECT o.order_id, c.customer_name
FROM orders AS o
JOIN customers FOR SYSTEM_TIME AS OF o.order_ts AS c
  ON o.customer_id = c.customer_id;
```

## How do you handle data skew?

In [ ]:
# Two-phase aggregation with a salted hot key: work spreads over subtasks, the total is unchanged.
import zlib
from collections import Counter, defaultdict

PARALLELISM, SALTS = 4, 4
events = [("tenant-big", 1)] * 1000 + [(f"tenant-{i}", 1) for i in range(40)]


def subtask(key: str) -> int:
    return zlib.crc32(key.encode()) % PARALLELISM          # stand-in for Flink's key-group hash


plain_load = Counter(subtask(key) for key, _ in events)

partial = defaultdict(int)                                    # phase 1: key + salt
for i, (key, value) in enumerate(events):
    partial[f"{key}#{i % SALTS}"] += value
salted_load = Counter()
for salted_key, value in partial.items():
    salted_load[subtask(salted_key)] += value

final = defaultdict(int)                                      # phase 2: strip the salt
for salted_key, value in partial.items():
    final[salted_key.split("#")[0]] += value

print("records per subtask, plain key: ", dict(sorted(plain_load.items())))
print("records per subtask, salted key:", dict(sorted(salted_load.items())))
assert final["tenant-big"] == 1000 and sum(final.values()) == len(events)
assert max(salted_load.values()) < max(plain_load.values())

**Skew is uneven key distribution, and keyed state magnifies it.** One subtask owns the hot key, so it does the work while the rest idle. The symptom is one subtask with high backpressure or long checkpoint duration while others are fast.

Fixes, in order of preference: pre-aggregate locally before the keyed shuffle so fewer records reach the hot key; salt the hot key into several buckets, aggregate, then combine; or rebalance the stream if the work is not truly key-scoped. For a heavy dimension on one key, broadcast it instead of joining by key.

**Common mistake:** raising parallelism to fix skew. With a single hot key, the key still maps to one subtask, so extra parallelism does nothing. Fix the key or the aggregation, not the slot count.

## How do you fix checkpoint timeouts?

**A checkpoint times out when it cannot finish inside `execution.checkpointing.timeout`.** The usual causes are large state, slow snapshot storage, barrier alignment stalled by backpressure, and long GC pauses. Start by reading the checkpoint history: duration, size, and alignment time tell you which cause applies.

Fixes map to causes. Incremental checkpoints and RocksDB reduce bytes written. Unaligned checkpoints remove the alignment wait under backpressure. A longer min-pause gives the job breathing room. Faster storage and smaller state reduce duration directly.

```yaml
execution.checkpointing.interval: 30s
execution.checkpointing.timeout: 10min
execution.checkpointing.min-pause: 10s
execution.checkpointing.unaligned.enabled: true
state.backend.incremental: true
```

**Common mistake:** raising the timeout as the only fix. If checkpoint size grows with state, longer timeouts just hide an unbounded growth problem until recovery itself becomes too slow.

## What is Flink CEP?

```java
Pattern<Event, ?> pattern = Pattern.<Event>begin("start")
    .where(evt -> evt.getAmount() > 10_000)
    .next("confirm")
    .where(evt -> evt.getAmount() > 10_000)
    .within(Time.minutes(5));
```

CEP lives in the Java/Scala `flink-cep` library; PyFlink has no `cep` module.

**CEP keeps a non-deterministic finite automaton per key.** Patterns are declared with `begin`, `next`, `followed_by`, `within`, and quantifiers. Matching state lives in keyed state, and partial matches are held until they complete or time out.

Time bounds matter. `within(Time.minutes(5))` stops a partial match from waiting forever. Without a bound, an incomplete pattern occupies state indefinitely. Timed-out partial matches can be routed to a side output for analysis.

**Common mistake:** running CEP without a time bound on a high-cardinality key. State grows as incomplete patterns accumulate; bound the pattern and clean up timed-out matches.

In [ ]:
# "Two transactions over 10,000 within 5 minutes" per account: keyed state plus a timeout,
# which is what a CEP pattern with within(5 minutes) compiles to.
WITHIN = 300
pending: dict[str, int] = {}                 # ValueState per account: timestamp of the first big txn


def process(account: str, ts: int, amount: int) -> str | None:
    first = pending.get(account)
    if first is not None and ts - first > WITHIN:
        del pending[account]                 # timer fired: the partial match timed out
        first = None
    if amount <= 10_000:
        return None
    if first is None:
        pending[account] = ts
        return None
    del pending[account]
    return f"ALERT {account}: big txns at {first}s and {ts}s"


stream = [("A", 0, 12_000), ("B", 10, 50), ("A", 120, 15_000), ("B", 20, 11_000), ("B", 400, 20_000)]
alerts = [a for a in (process(*e) for e in stream) if a]
print(alerts)
assert alerts == ["ALERT A: big txns at 0s and 120s"]      # B's second big txn came after 5 minutes

## Can Flink lose data?

**Loss comes from three places, and checkpointing removes the first two.** 1) No checkpointing or a non-replayable source, so a failure cannot rewind. 2) A checkpoint that never completes, so there is no recovery point. 3) A sink that acknowledges before the data is durable, or a source that discards records before the checkpoint commits.

Exactly-once requires all three to align: replayable source, completed checkpoints, and a transactional or idempotent sink. At-least-once can duplicate but should not lose, provided the source can replay.

**Common mistake:** assuming exactly-once on internal state protects an external side effect. A non-transactional external write can be lost or duplicated even when the Flink state is consistent.

## How do you handle late data?

Watermarks plus allowed lateness define the window's closing and grace period.

```python
stream.key_by(lambda e: e["user"]).window(
    TumblingEventTimeWindows.of(Time.minutes(5))
).allowed_lateness(120_000).side_output_late_data(late_tag).process(MyWindowFn())   # 2 minutes in ms
```

Watermark delay, allowed lateness, and the side output, simulated:

In [ ]:
# Tumbling 1-minute windows with a 5 s watermark delay, 30 s allowed lateness, and a side output.
DELAY, LATENESS, SIZE = 5, 30, 60
results: dict[int, int] = {}
emitted: list[tuple[int, int, str]] = []
late_side_output: list[int] = []
watermark = float("-inf")

for ts in [10, 20, 65, 30, 70, 95, 40, 130, 50]:          # event times in arrival order
    window_end = ts - ts % SIZE + SIZE
    if window_end + LATENESS <= watermark:
        late_side_output.append(ts)                        # too late even for allowed lateness
        continue
    results[window_end] = results.get(window_end, 0) + 1
    if window_end <= watermark:                            # already fired: emit an update
        emitted.append((window_end, results[window_end], "update"))
    watermark = max(watermark, ts - DELAY)
    for end in sorted(results):
        if end <= watermark and not any(e == end for e, _, _ in emitted):
            emitted.append((end, results[end], "final"))

print("emitted:", emitted)
print("side output:", late_side_output)
assert emitted[:2] == [(60, 2, "final"), (60, 3, "update")]   # 30 was late but within lateness
assert late_side_output == [40, 50]                            # arrived after watermark >= 60 + 30

**Three levers, from cheapest to most correct.** Enlarge the watermark delay so fewer records are late. Allow lateness on the window so a late record still updates the result and re-emits it. Route records that exceed the allowed lateness to a side output for a separate reconciliation job.

The choice is a business decision. Billing may accept a long delay at the cost of latency. A live dashboard may prefer an early approximate result and no corrections. Record the policy where the pipeline is defined.

**Common mistake:** setting `allowed_lateness` without a sink that can handle updates. The corrected result is emitted, but an append-only sink turns it into a duplicate row instead of an update.

## When do you use aggregate versus process?

```python
# incremental: cheap in memory, no access to individual records
keyed.window(TumblingEventTimeWindows.of(Time.minutes(5))).aggregate(SumAmount())

# full window: needed for Top-N, sorting, or custom timers
keyed.window(TumblingEventTimeWindows.of(Time.minutes(5))).process(TopNPerRegion())
```

**Aggregate when the result is a function of the values; process when it depends on time and order.** `reduce`, `aggregate`, and `AggregatingState` keep one value per key, so state is bounded by the number of keys. `KeyedProcessFunction` exposes timers and lets you buffer records, which is necessary for pattern detection, sessionization, and custom emission.

The cost difference is large. An aggregate over a million keys holds a million small values. A process function that appends each record to a `ListState` holds every record until you clear it.

**Common mistake:** choosing `process` for convenience and then discovering the state cannot recover in reasonable time. If the logic is a running total, an aggregate is both simpler and cheaper.

## How does checkpointing work in Flink?

```python
from pyflink.datastream import CheckpointingMode, ExternalizedCheckpointCleanup

env.enable_checkpointing(10_000, CheckpointingMode.EXACTLY_ONCE)
cfg = env.get_checkpoint_config()
cfg.set_min_pause_between_checkpoints(5_000)
cfg.set_checkpoint_timeout(60_000)
cfg.set_max_concurrent_checkpoints(1)
cfg.enable_externalized_checkpoints(ExternalizedCheckpointCleanup.RETAIN_ON_CANCELLATION)
```

Key knobs: interval (latency vs overhead), timeout (how long before a checkpoint is abandoned), min-pause (breathing room between checkpoints), and unaligned checkpoints (skip barrier alignment under backpressure).

**Alignment versus unaligned.** In an aligned checkpoint, an operator that receives a barrier on one input stops processing that input until every input has delivered its barrier, so the snapshot is a consistent cut. Under backpressure, that wait can dominate checkpoint duration.

Unaligned checkpoints let the barrier overtake buffered records. In-flight data is included in the snapshot, so the alignment wait disappears, at the cost of larger checkpoints. Exclude the alignment stall first, then decide whether the extra size is acceptable.

**Common mistake:** enabling unaligned checkpoints on a job whose real problem is slow state writes. Unaligned fixes barrier stalls, not storage throughput; checkpoint size grows instead.

### Try it - checkpoint and replay

Why restoring a checkpoint gives exactly-once *state*: the source offset and operator state are captured at the same barrier.

In [ ]:
# Checkpoint and replay: source offset and operator state are snapshotted together, so a crash
# rewinds both and the keyed counts end exactly as if nothing failed.
import copy

topic = [("u1", 1), ("u2", 1), ("u1", 1), ("u3", 1), ("u1", 1), ("u2", 1), ("u1", 1)]
CHECKPOINT_EVERY, CRASH_AT = 3, 5


def run_job(crash_at: int | None) -> dict[str, int]:
    state, offset = {}, 0
    snapshot = {"offset": 0, "state": {}}
    crashed = False
    while offset < len(topic):
        if offset == crash_at and not crashed:
            crashed = True                                   # task fails: restore the last checkpoint
            offset, state = snapshot["offset"], copy.deepcopy(snapshot["state"])
            continue
        key, value = topic[offset]
        state[key] = state.get(key, 0) + value
        offset += 1
        if offset % CHECKPOINT_EVERY == 0:                   # barrier passes: snapshot offset + state
            snapshot = {"offset": offset, "state": copy.deepcopy(state)}
    return state


clean, recovered = run_job(crash_at=None), run_job(crash_at=CRASH_AT)
print("no failure:", clean, "| crash and restore:", recovered)
assert clean == recovered == {"u1": 4, "u2": 2, "u3": 1}

## What are savepoints and how do you use them?

**Savepoints are a compatibility contract.** They are triggered manually, owned by you, and used for upgrades, rescaling, and migration. Restoring a savepoint checks that every operator in the new job maps to state in the savepoint. Renaming a UID without care breaks that mapping.

Always assign stable `uid` values to stateful operators before the first production run. With `uid`s fixed, you can change parallelism, swap a source, or reorder operators and still restore.

```bash
flink savepoint <jobId> s3://acme-flink/savepoints/
flink run -s s3://acme-flink/savepoints/savepoint-abc123 -p 32 orders.jar
```

**Common mistake:** relying on `--allowNonRestoredState` to skip mismatches. It hides unintended state loss; use it only when you are certain the dropped state is no longer needed.

## How do you configure the state backend?

```yaml
# flink-conf.yaml
state.backend: rocksdb
state.backend.incremental: true
state.checkpoints.dir: s3://acme-flink/checkpoints/
state.savepoints.dir: s3://acme-flink/savepoints/
state.backend.rocksdb.memory.managed: true
```

```python
from pyflink.datastream import StreamExecutionEnvironment, EmbeddedRocksDBStateBackend

env = StreamExecutionEnvironment.get_execution_environment()
env.set_state_backend(EmbeddedRocksDBStateBackend(enable_incremental_checkpointing=True))
```

**Configure at the cluster level and optionally per job.** `state.backend` selects heap (`hashmap`) or `rocksdb`. Incremental checkpoints require RocksDB and checkpoint storage that supports them, which cloud object stores and HDFS do. RocksDB memory is governed by managed memory so it does not fight the JVM heap.

After changing the backend, the next checkpoint must be a full one in some migration paths. Plan it as a savepoint-and-restart operation, not a hot switch.

**Common mistake:** enabling `state.backend.incremental` while writing checkpoints to a filesystem that does not support the required operations, then seeing every checkpoint fail. Verify the storage first.

## How do you build an exactly-once Kafka to Kafka pipeline?

End-to-end exactly-once needs three things: checkpointing enabled, a replayable source that stores offsets in Flink state, and a transactional sink that only commits when a checkpoint completes.

```python
from pyflink.common import WatermarkStrategy
from pyflink.common.serialization import SimpleStringSchema
from pyflink.datastream import StreamExecutionEnvironment, CheckpointingMode
from pyflink.datastream.connectors.kafka import (
    KafkaSource, KafkaSink, KafkaOffsetsInitializer,
    KafkaRecordSerializationSchema, DeliveryGuarantee,
)

env = StreamExecutionEnvironment.get_execution_environment()
env.enable_checkpointing(10_000, CheckpointingMode.EXACTLY_ONCE)

source = (
    KafkaSource.builder()
    .set_bootstrap_servers("broker1:9092")
    .set_topics("orders")
    .set_group_id("orders-flink")
    .set_starting_offsets(KafkaOffsetsInitializer.committed_offsets())
    .set_value_only_deserializer(SimpleStringSchema())
    .build()
)

sink = (
    KafkaSink.builder()
    .set_bootstrap_servers("broker2:9092")
    .set_record_serializer(
        KafkaRecordSerializationSchema.builder()
        .set_topic("orders-enriched")
        .set_value_serialization_schema(SimpleStringSchema())
        .build()
    )
    .set_delivery_guarantee(DeliveryGuarantee.EXACTLY_ONCE)
    .set_transactional_id_prefix("orders-flink-")
    .build()
)

stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "kafka-orders")
stream.map(lambda v: v.upper()).sink_to(sink)
env.execute("orders-exactly-once")
```

**The three ingredients, in order.** A Kafka source that stores offsets in Flink state and starts from committed or earliest offsets. Checkpointing enabled with `EXACTLY_ONCE` mode. A Kafka sink with `DeliveryGuarantee.EXACTLY_ONCE`, which uses Kafka transactions committed only when the checkpoint completes.

Because Kafka transactions are per checkpoint, the sink must be configured with a transaction timeout longer than the checkpoint interval plus recovery time, and consumers must read with `read_committed` isolation to avoid seeing uncommitted records.

**Common mistake:** a consumer that defaults to `read_uncommitted`. It can see records from a transaction that is later aborted, breaking the exactly-once illusion downstream.

## How do you do windowing in Flink SQL?

Flink SQL uses table-valued window functions, which are composable with normal `GROUP BY` and joins.

```sql
-- Tumbling window: one row per minute per region
SELECT window_start, window_end, region, SUM(amount) AS revenue
FROM TABLE(TUMBLE(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' MINUTE))
GROUP BY window_start, window_end, region;

-- Sliding (HOP) window: emitted every minute over a 10-minute span
SELECT window_start, window_end, region, COUNT(*) AS orders
FROM TABLE(HOP(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' MINUTE, INTERVAL '10' MINUTE))
GROUP BY window_start, window_end, region;

-- Cumulative window: running totals for the day
SELECT window_start, window_end, region, SUM(amount) AS revenue
FROM TABLE(CUMULATE(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' HOUR, INTERVAL '1' DAY))
GROUP BY window_start, window_end, region;
```

**Window TVFs return extra columns.** `TUMBLE`, `HOP`, and `CUMULATE` wrap the table and expose `window_start` and `window_end`. Group by those columns plus your key. The result is emitted as the watermark advances past the window end.

Sinks decide the update semantics. A windowed aggregate is insert-only per window, so append sinks work. A non-windowed `GROUP BY` produces an updating changelog and needs an upsert-capable sink keyed by the group columns.

**Common mistake:** forgetting that a HOP window emits each record into multiple overlapping windows. The work and output multiply by the number of windows each record falls into.

## How do you handle event time and watermarks in PyFlink?

Assign timestamps and a watermark strategy at the source, so every downstream operator has a consistent notion of event time.

```python
from pyflink.common import Duration, WatermarkStrategy
from pyflink.common.watermark_strategy import TimestampAssigner
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

class EventTimestamps(TimestampAssigner):
    def extract_timestamp(self, event, record_timestamp):
        return event["event_ts"]

watermarks = (
    WatermarkStrategy
    .for_bounded_out_of_orderness(Duration.of_seconds(5))
    .with_idleness(Duration.of_minutes(1))
)

stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "kafka-orders")
stream = stream.assign_timestamps_and_watermarks(
    watermarks.with_timestamp_assigner(EventTimestamps())
)
keyed = stream.key_by(lambda e: e["user_id"])
```

**Assign watermarks as early as possible.** Ideally the source carries a `WatermarkStrategy`, so timestamp assignment happens before any repartition and every downstream operator sees the same event time. Use `with_timestamp_assigner` when the timestamp is not a plain field.

For Kafka, prefer `KafkaRecordDeserializationSchema` that emits records with timestamps, or assign watermarks right after the source. Watermarks assigned after a shuffle are computed on the shuffled stream, which is usually still fine but harder to reason about.

**Common mistake:** assigning watermarks after a `key_by` or join. The source then has no event time, so any window or timer upstream of that point silently uses processing time or fails.

## What are Flink's trade-offs and when should you not use it?

+ **Performance:** millisecond latency and true event-time semantics, but throughput per core is lower than Spark's micro-batch and JVM/GC tuning matters.
+ **Resource contention:** state size, checkpoint duration, and TaskManager memory are tightly coupled; RocksDB state spills to local disk and a slow disk becomes a checkpoint bottleneck.
+ **Exactly-once:** guaranteed only when the source is replayable and the sink supports transactions or idempotent upserts. An external JDBC write is not transactional unless you make it idempotent.
+ **Operations:** savepoints, state schema evolution, and watermark/backpressure diagnosis require real operational maturity.

## Tuning: diagnosing backpressure and slow operators

**Start with the job graph, not the logs.** The Flink UI colors each operator by backpressure. A backpressured operator is waiting on something downstream; the bottleneck is the first operator downstream that is busy and not backpressured. Read the two metrics together: `busyTime` tells you the operator is doing work, `backPressuredTime` tells you it cannot write its output.

The ratio is what matters. `busyTime / (busyTime + backPressuredTime + idleTime)` near 1 with a low backpressure ratio means this operator is the constraint. A high backpressure ratio with low busy means look further downstream. Idle time near 1 means the operator is starved by its input.

**Worked example:** a window operator shows `backPressuredTime` at 85 percent, while the Kafka sink after it shows `busyTime` at 95 percent and near-zero backpressure. The sink is the bottleneck; the window is a victim. Adding parallelism to the window changes nothing.

**Checkpoint metrics separate two failure modes.** Slow checkpoints with small size point at storage or alignment. Slow checkpoints with large and growing size point at state that is never cleaned. The checkpoint history in the UI shows duration, size, and alignment time per completed checkpoint, which is enough to tell them apart.

When alignment time dominates, barriers are stuck behind buffered records, which is the signature of backpressure. When the synchronous phase dominates, an operator is doing too much work while holding up its own snapshot. When the asynchronous phase dominates, the snapshot storage is too slow.

**Common mistake:** treating every red operator in the UI as a problem to fix. Most red operators are just waiting. Fix the one at the end of the chain, then re-read the graph.

## Tuning: state backends and checkpointing

**Choose the backend from state size and churn.** `HashMapStateBackend` keeps state as JVM objects: fastest access, full snapshots, bounded by heap. `EmbeddedRocksDBStateBackend` keeps state in a local embedded database: larger-than-memory state and incremental checkpoints, at the cost of serialization and local disk I/O.

Incremental checkpoints write only files changed since the last checkpoint. They require RocksDB and durable storage that supports the operations, such as HDFS or S3. Heap state cannot be incremental; every checkpoint copies everything.

**Worked example:** 40 GB of keyed state on the heap means a 40 GB snapshot every checkpoint and a heap that must hold it. The same state on RocksDB with incremental checkpoints typically uploads only a few hundred MB per checkpoint, because most files are unchanged.

**Common mistake:** adding state TTL only after checkpoints become slow. Attach it when the state is designed, and choose an expiry that matches the business rule.

**Tune the checkpoint cadence as a budget.** Overhead is roughly checkpoint duration divided by checkpoint interval. A 30-second interval with a 10-second checkpoint spends a third of the time snapshotting. Min-pause keeps two checkpoints from running back to back. Concurrent checkpoints let a slow one overlap the next, which increases resource use but hides latency.

Unaligned checkpoints remove the alignment wait under backpressure and include in-flight data in the snapshot, so they grow the checkpoint. Enable them when alignment dominates; disable them when checkpoint size is the problem.

```yaml
state.backend: rocksdb
state.backend.incremental: true
state.checkpoints.dir: s3://acme-flink/checkpoints/
execution.checkpointing.interval: 30s
execution.checkpointing.timeout: 10min
execution.checkpointing.min-pause: 10s
execution.checkpointing.max-concurrent-checkpoints: 1
execution.checkpointing.unaligned.enabled: true
execution.checkpointing.tolerable-failed-checkpoints: 3
```

**Common mistake:** setting the interval shorter to get a better recovery point while checkpoints already take longer than the interval. Overlapping snapshots compete with processing and throughput falls.

## Tuning: parallelism, slots, and resources

**Set parallelism per stage from its real constraint.** Source parallelism cannot exceed Kafka partitions. Keyed stages scale with distinct keys, not with records. Stateless stages scale with cores. The cluster must provide slots for the largest stage, so an oversized stage holds the whole job back.

Slots per TaskManager are an isolation choice. One slot per core is a common start, but memory-heavy pipelines need more slots per JVM to amortize framework memory, and CPU-heavy pipelines want fewer to avoid contention. Managed memory is the share available to RocksDB and batch operators.

```yaml
parallelism.default: 16
taskmanager.numberOfTaskSlots: 4
taskmanager.memory.process.size: 8192m
taskmanager.memory.managed.fraction: 0.4
jobmanager.memory.process.size: 2048m
```

```bash
# command-line parallelism is the default for the submission
flink run -d -p 16 -c com.acme.OrdersJob orders.jar
```

## Tuning: source and sink throughput

**Match source parallelism to partitions.** A Kafka source subtask reads one or more partitions; more source subtasks than partitions leaves readers idle. Check the partitions, not the topics, before setting source parallelism.

Watermarks need idleness to tolerate silent partitions, and the auto-watermark interval controls how often watermarks are emitted. A very small interval adds overhead; a very large one delays window firing.

```yaml
pipeline.auto-watermark-interval: 200ms
```

**Common mistake:** reading a topic with 6 partitions at source parallelism 24 and expecting 4x throughput. Only 6 subtasks do work; the rest wait and consume slots the downstream stages could use.

## Things to remember: key numbers and limits

| Topic | Fact | Why it matters |
|---|---|---|
| Event vs processing time | event time needs watermarks to know when a window can close | results are correct under disorder only with event time |
| Watermark | a timestamp heuristic: 'no events older than T should arrive' | too aggressive drops late data; too loose delays every window |
| State | keyed/operator state held in a configured state backend (HashMap/RocksDB) | state size dictates the backend and checkpoint duration |
| Checkpoints -> savepoints | checkpoints are automatic for recovery; savepoints are manual, versioned snapshots for upgrades/migrations | deploy new code against a savepoint, not cold |
| Consistency | checkpoint barrier alignment gives exactly-once internal state; end-to-end needs a transactional or idempotent sink | 'exactly-once' is only as strong as the sink |
| Parallelism vs slots | parallelism = subtasks; slots = TaskManager capacity slots that share JVM memory | parallelism beyond slots queues; slots beyond parallelism waste JVMs |
| APIs | DataStream for streaming control, Table/SQL for declarative pipelines; DataSet API is legacy | new work is DataStream or Table |
| Backpressure | downstream slowness propagates up the topology | the backpressure ratio per operator is the triage metric |

## Real world use cases for Apache Flink

### 1. Real-time fraud detection

A payments company scores every card transaction within milliseconds and blocks suspicious ones. ~8,000 transactions/second (~700 million/day), 30-day rolling per-account state. Kafka carries transactions keyed by `account_id`; Flink keeps per-account velocity and amount state in RocksDB and writes alerts to Kafka and a case-management API.

```python
keyed = transactions.key_by(lambda t: t["account_id"])
alerts = keyed.process(VelocityDetector())   # timers + ValueState per account
alerts.sink_to(kafka_sink("fraud.alerts"))
```

Why: event time plus keyed state gives deterministic, replayable scoring, and exactly-once avoids double-blocking a customer after a failure.

### 2. Continuous lakehouse ingestion from CDC

A retailer streams Debezium change events into an Iceberg table with upserts. ~120,000 events/second, ~40 TB of current-state data, 5-minute checkpoints. Kafka source with committed offsets, Flink SQL `MERGE` (or the Iceberg sink), and hourly Iceberg compaction.

```sql
INSERT INTO lakehouse.customers
SELECT customer_id, name, email, updated_at
FROM cdc_customers;
```

```python
env.enable_checkpointing(300_000, CheckpointingMode.EXACTLY_ONCE)
```

Why: checkpointed Kafka offsets plus a transactional Iceberg sink make ingestion exactly-once, and Flink absorbs bursty CDC without a nightly rebuild.

### 3. Sessionization and product analytics

A media platform groups viewer events into sessions and computes engagement metrics. ~25,000 events/second, 10-minute session gaps, late-arriving mobile events up to 2 minutes old. Session windows with allowed lateness and a side output for late records.

```python
sessions = (
    events.key_by(lambda e: e["user_id"])
    .window(EventTimeSessionWindows.with_gap(Time.minutes(10)))
    .allowed_lateness(120_000)   # 2 minutes in ms
    .side_output_late_data(late_tag)
    .process(SessionMetrics())
)
```

Why: session windows match the business concept directly, and allowed lateness plus a side output turns "late" into a handled case instead of a silent drop.

### 4. Real-time alerting on infrastructure metrics

A SaaS provider detects anomalies in host and application metrics. ~500,000 samples/second, 1-second resolution, 15-minute sliding windows. Flink aggregates metrics, compares against a broadcast model, and emits alerts.

```python
windowed = (
    metrics.key_by(lambda m: m["host_id"])
    .window(SlidingEventTimeWindows.of(Time.minutes(15), Time.minutes(1)))
    .aggregate(MetricStats())
)
alerts = windowed.connect(model_stream.broadcast(state_descriptor)).process(AnomalyFn())
```

Why: a broadcast state stream lets the detection model update without redeploying the job, and sliding windows give continuous coverage instead of fixed buckets.

### 5. Exactly-once enrichment pipeline to Kafka

A logistics company enriches shipment events with customer and route data and republishes them. ~15,000 events/second, JDBC dimensions with a 10-minute cache, exactly-once output. Async lookup join for the dimension, Kafka source and transactional Kafka sink.

```python
env.enable_checkpointing(10_000, CheckpointingMode.EXACTLY_ONCE)
enriched = orders.key_by(lambda o: o["customer_id"]).map(AsyncDimLookup())
enriched.sink_to(exactly_once_kafka_sink("shipments-enriched"))
```

Why: async I/O keeps the external lookup off the operator thread, and transactional Kafka writes plus checkpointed offsets give true end-to-end exactly-once.

## Flink exercises

These exercises use only the Python standard library. Each one models a sizing or diagnosis decision you make in a real Flink job.

### Try it 1. Choose source parallelism from Kafka partitions

**Task.** Given a topic's partition count and a target parallelism from throughput, return the useful source parallelism. A Kafka source subtask can only read whole partitions, so parallelism cannot exceed the partition count.

**Hint.** The answer is the smaller of the partition count and the target parallelism. Also flag when the target is wasted because partitions are the binding constraint.

**Expected.** 24 partitions with target 16 gives 16 usable subtasks. 6 partitions with target 24 gives 6, and 18 subtasks would be idle.

In [ ]:
def source_parallelism(partitions, target):
    """Useful Kafka source parallelism: never more than the partitions."""
    return min(partitions, target)


def idle_subtasks(partitions, target):
    """Subtasks that would sit idle if parallelism is the target."""
    return max(0, target - partitions)


assert source_parallelism(24, 16) == 16
assert source_parallelism(6, 24) == 6
assert idle_subtasks(6, 24) == 18

print("24 partitions, target 16 ->", source_parallelism(24, 16))
print("6 partitions, target 24 ->", source_parallelism(6, 24),
      "(idle subtasks:", idle_subtasks(6, 24), ")")

### Try it 2. Compute checkpoint overhead from interval and size

**Task.** Given the checkpoint interval in milliseconds and a measured checkpoint duration, compute the fraction of wall-clock time the job spends snapshotting, including the min-pause.

**Hint.** Overhead is duration divided by the effective cycle, where the cycle is the interval plus min-pause when min-pause forces a gap. Keep the result as a fraction.

**Expected.** A 30-second interval with a 4.5-second checkpoint and 5 seconds of min-pause gives 4.5 / 35 = about 13 percent overhead.

In [ ]:
def checkpoint_overhead(interval_ms, duration_ms, min_pause_ms=0):
    """Fraction of time spent on checkpoints, including the min-pause."""
    cycle = interval_ms + min_pause_ms
    return duration_ms / cycle


overhead = checkpoint_overhead(30_000, 4_500, 5_000)
assert abs(overhead - 4_500 / 35_000) < 1e-9
assert overhead < 0.30

print(f"overhead = {overhead:.1%}")
print("within budget" if overhead < 0.20 else "consider a longer interval or smaller state")

### Try it 3. Detect backpressure from busy and idle metrics

**Task.** Given per-operator busy, backpressured, and idle times, compute the busy ratio and backpressure ratio, then name the bottleneck. The bottleneck is the operator that is busy and not itself backpressured.

**Hint.** An operator with backpressure is waiting downstream. The culprit is upstream of the backpressured ones, has a high busy ratio, and a low backpressure ratio.

**Expected.** In the sample, the window is heavily backpressured and the sink is busy with almost no backpressure, so the sink is the bottleneck.

In [ ]:
def ratios(busy, backpressured, idle):
    total = busy + backpressured + idle
    return busy / total, backpressured / total


operators = [
    ("source", 120, 20, 860),
    ("window", 150, 810, 40),
    ("sink", 920, 30, 50),
]


def find_bottleneck(ops):
    busy_ratio, bp_ratio = {}, {}
    for name, busy, bp, idle in ops:
        busy_ratio[name], bp_ratio[name] = ratios(busy, bp, idle)
    candidates = [n for n in busy_ratio if busy_ratio[n] > 0.8 and bp_ratio[n] < 0.1]
    return candidates[-1] if candidates else None


assert find_bottleneck(operators) == "sink"
for name, busy, bp, idle in operators:
    br, pr = ratios(busy, bp, idle)
    print(f"{name:8s} busy={br:.0%} backpressure={pr:.0%}")
print("bottleneck ->", find_bottleneck(operators))

### Try it 4. Size state from key count and per-key bytes

**Task.** Estimate heap state size from the number of keys and bytes per key, apply an overhead factor, and recommend a state backend. Use RocksDB when the estimate exceeds a heap budget.

**Hint.** Size = keys times bytes per key times an overhead factor (1.5 is a reasonable default for object headers and map entries). Compare it with the heap budget.

**Expected.** 2 million keys at 400 bytes with 1.5x overhead is 1.2 GB. With a 1 GB budget the recommendation is RocksDB.

In [ ]:
def state_size_bytes(keys, bytes_per_key, overhead=1.5):
    return int(keys * bytes_per_key * overhead)


def recommend_backend(keys, bytes_per_key, heap_budget_bytes):
    size = state_size_bytes(keys, bytes_per_key)
    return ("rocksdb" if size > heap_budget_bytes else "hashmap"), size


assert state_size_bytes(2_000_000, 400) == 1_200_000_000
backend, size = recommend_backend(2_000_000, 400, 1_000_000_000)
assert backend == "rocksdb"

print(f"estimated state = {size / 1e9:.2f} GB")
print("backend ->", backend)

### Try it 5. Set watermark delay from late-event distribution

**Task.** Given a list of observed lateness values in seconds, pick a watermark delay that covers the 99th percentile, rounded up to the next whole second. Report how many events it still misses.

**Hint.** Use a nearest-rank percentile: index = ceil(0.99 * n) - 1 on the sorted list. The delay is that value rounded up.

**Expected.** With lateness mostly under 4 seconds but a tail at 8 seconds, the delay is 8 seconds and covers at least 99 percent of events.

In [ ]:
import math


def percentile(sorted_values, p):
    k = max(0, math.ceil(p * len(sorted_values)) - 1)
    return sorted_values[k]


def watermark_delay(sorted_lateness_s, p=0.99):
    """Whole-second watermark delay covering the p-th percentile."""
    return math.ceil(percentile(sorted_lateness_s, p))


lateness = sorted([0, 1, 2, 1, 3, 2, 2, 1, 0, 4, 2, 3, 2, 1, 1, 2, 8, 2, 3, 2])
delay = watermark_delay(lateness)
covered = sum(1 for v in lateness if v <= delay) / len(lateness)

assert delay == 8
assert covered >= 0.99

print("watermark delay =", delay, "seconds")
print(f"covers {covered:.0%} of observed lateness")

### Try it 6. Plan a savepoint and upgrade

**Task.** Before an upgrade, compare the stateful operator UIDs in the running job with the new job. Restoring is safe when every old UID still exists. Report missing UIDs and whether the new job adds stateful operators with new UIDs.

**Hint.** Use sets. Missing old UIDs mean restore will fail unless you accept state loss with `--allowNonRestoredState`.

**Expected.** The sample plan keeps all old UIDs and adds one new one, so it restores cleanly without allowing non-restored state.

In [ ]:
def plan_upgrade(old_uids, new_uids):
    missing = sorted(set(old_uids) - set(new_uids))
    added = sorted(set(new_uids) - set(old_uids))
    safe = not missing
    return {
        "missing": missing,
        "added": added,
        "restore_ok": safe,
        "needs_allow_non_restored": not safe,
    }


old = ["source-orders", "window-revenue", "sink-warehouse"]
new = ["source-orders", "window-revenue", "sink-warehouse", "sink-alerts"]
plan = plan_upgrade(old, new)

assert plan["restore_ok"] is True
assert plan["added"] == ["sink-alerts"]
assert plan["missing"] == []

print("savepoint restore ok:", plan["restore_ok"])
print("new stateful operators:", plan["added"])

### Try it 7. Estimate network buffer pressure from records per second

**Task.** Estimate how much data is in flight on an output channel from the record rate, average record size, and buffer timeout. Compare it with the network memory available per TaskManager.

**Hint.** In-flight bytes = records per second times bytes per record times buffer timeout in seconds. If it exceeds the network budget, raise network memory or lower the timeout.

**Expected.** 50k records/s at 500 bytes with a 100 ms timeout is 2.5 MB in flight, comfortably under a 48 MB budget.

In [ ]:
def in_flight_bytes(records_per_s, bytes_per_record, buffer_timeout_ms):
    return records_per_s * bytes_per_record * (buffer_timeout_ms / 1000.0)


def pressure_ratio(records_per_s, bytes_per_record, buffer_timeout_ms, network_bytes):
    return in_flight_bytes(records_per_s, bytes_per_record, buffer_timeout_ms) / network_bytes


inflight = in_flight_bytes(50_000, 500, 100)
assert inflight == 2_500_000

ratio = pressure_ratio(50_000, 500, 100, 48 * 1024 * 1024)
assert ratio < 0.2

print(f"in-flight bytes = {inflight / 1e6:.1f} MB")
print(f"network pressure = {ratio:.1%} of budget")